# 🎮 Colab Gaming PC (L4 GPU / High-Memory) - クラウドゲーミング＆サーバー統合環境

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/9z2fqjckjv-bot/ColabTool/blob/main/GamingPC/GamingPC_Colab.ipynb)

Google Colab の強力な **NVIDIA L4 GPU (24GB VRAM) & ハイメモリ (53GB RAM)** を活用し、本格的なクラウドゲーミングワークステーションおよび自立型 Minecraft サーバーをワンクリックで構築・稼働させる自動化ノートブックです。

---

### 🌟 本バージョンの主な進化・機能
- **🇯🇵 日本語 UI & 日本標準時 (JST / 東京)**: Xfce4 デスクトップ全体・時計・ロケール・Google Noto CJK/カラー絵文字フォントを完全日本語化。
- **💻 MacBook 最適化解像度 (16:10 / 1680x1050)**: MacBook の画面比率に合わせてジャストフィット。はみ出しや黒帯なく快適に操作可能。
- **⚡ Web Audio API 超低遅延・音同期エンジン**: 従来のブラウザ遅延を完全排除。PCM 16bit 44.1kHz で映像と音が 0.1秒未満で同期（YouTube・ゲームのリップシンク対応）。
- **🛡️ Steam & 一般ユーザー権限の完全整備**: Colab の root 制限を回避する一般ユーザー (`colabuser`) とセーフティラッパーを自動構成。Steam や Discord が警告なく一発起動。
- **🌐 Google Chrome マルチプロファイル & 充実のランチャー**:
  - **Chrome Profile 1 / 2 / 3**: 個別の独立プロファイルでタブやアカウントを使い分け。
  - **Steam**: Hoi4、ChillwithYou などをバックグラウンド実行・プレイ。
  - **Minecraft (Prism Launcher)**: Modpack や各バージョンをワンクリック管理。
  - **CurseForge / Xbox Cloud Gaming / Roblox / Discord / OBS Studio**
- **⛏️ 独立稼働型 Minecraft サーバー**: PaperMC 1.21.1 をバックグラウンドデーモンとして起動。playit.gg による外部接続対応。
- **💾 Google Drive 完全データ永続化**: ワールド、セーブデータ、録画、プロファイルを自動バックアップ。

---
## 📍 ステップ 0: リージョン確認＆東京リージョン（Tokyo, JP）最適化

現在の接続先データセンターの所在地（リージョン）と Ping 遅延をチェックします。  
**東京リージョン（`Tokyo, JP` / `asia-northeast1`）** に接続することで、画面操作やゲームプレイの遅延（Ping 5〜15ms）が最小になります。

In [ ]:
#@title 📍 ステップ 0: リージョン確認＆東京リージョン判定
import urllib.request
import json
import time
from IPython.display import display, HTML

print("🔍 接続先データセンターのリージョン情報を照会中...")
region_info = {}
try:
    req = urllib.request.Request("https://ipinfo.io/json", headers={"User-Agent": "curl/7.68.0"})
    with urllib.request.urlopen(req, timeout=5) as response:
        region_info = json.loads(response.read().decode())
except Exception as e:
    region_info = {"city": "不明", "country": "不明", "org": str(e)}

city = region_info.get("city", "不明")
country = region_info.get("country", "不明")
ip = region_info.get("ip", "-")
org = region_info.get("org", "Google LLC")

is_tokyo = (city.lower() == "tokyo" or country == "JP")

status_color = "#10b981" if is_tokyo else "#f59e0b"
status_title = "🎉 東京リージョン（Tokyo, JP）で接続中！" if is_tokyo else f"⚠️ 現在は {city}, {country} リージョンで接続されています"
status_msg = "遅延（Ping 5〜15ms）が最小のベストな環境です。そのままステップ 1 へ進んでください。" if is_tokyo else (
    "シンガポールや台湾などのアジア近郊リージョンの場合、通常利用（約60ms）は可能ですが、より低遅延な東京リージョンを狙うには上部メニューの「ランタイム」➔「ランタイムを接続解除して削除」を行い、再接続をお試しください。"
)

display(HTML(f'''
<div style="background: #0f172a; padding: 22px; border-radius: 12px; border: 1px solid #334155; border-left: 6px solid {status_color}; font-family: sans-serif;">
  <h3 style="color: {status_color}; margin: 0 0 10px 0; font-size: 18px;">{status_title}</h3>
  <p style="color: #e2e8f0; font-size: 14px; margin-bottom: 12px;">{status_msg}</p>
  <div style="background: #1e293b; padding: 12px; border-radius: 8px; font-size: 13px; color: #94a3b8;">
    <strong>IP:</strong> {ip} | <strong>所在地:</strong> {city}, {country} | <strong>ネットワーク:</strong> {org}
  </div>
</div>
'''))


---
## 🚀 ステップ 1: 【ワンクリック起動】Gaming PC & 独立Minecraftサーバー

下のセルを実行するだけで、**日本語環境・MacBook 16:10 解像度・超低遅延音同期・全ゲームランチャー** が全自動で立ち上がります。  
バックグラウンドで独立稼働するため、**約1〜2分でこのセルの実行は正常完了**します。

> [!TIP]
> 実行完了後、セル下部に表示される **「🚀 今すぐデスクトップを開く」** ボタンをクリックしてください！

In [ ]:
#@title 🎮 ステップ 1: 【ワンクリック起動】Gaming PC & 独立Minecraftサーバー
import os
import sys
import json
import subprocess
import time
from IPython.display import display, HTML
from google.colab.output import eval_js

#@markdown ### 💻 画面解像度の選択 (MacBook 向け 16:10 推奨)
SCREEN_RES = "1680x1050" #@param ["1680x1050", "1920x1200", "1920x1080", "1440x900"]

#@markdown ### 🌐 リモート接続方式の選択
REMOTE_METHOD = "\u4e21\u65b9\u8d77\u52d5 (noVNC + Chrome Remote Desktop \u63a8\u5968)" #@param ["noVNC (Webブラウザで即座にアクセス・設定不要)", "Chrome Remote Desktop (最高画質・低遅延・ゲーム推奨)", "両方起動 (noVNC + Chrome Remote Desktop 推奨)"]
CRD_AUTH_CODE = "" #@param {type:"string"}
CRD_PIN = "123456" #@param {type:"string"}

#@markdown ### ⛏️ Minecraftサーバー設定
START_MINECRAFT_SERVER = True #@param {type:"boolean"}
MC_RAM_GB = 8 #@param {type:"slider", min:4, max:16, step:2}

#@markdown ### 💾 Google Drive バックアップ
MOUNT_GOOGLE_DRIVE = True #@param {type:"boolean"}

print("==================================================")
print("  🎮 Colab Gaming PC & 独立Minecraftサーバー 起動")
print("==================================================")

# 1. Google Drive の自動マウント
if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        if not os.path.exists("/content/drive/MyDrive"):
            print("📂 Google Drive をマウント中...")
            drive.mount("/content/drive")
            print("✅ Google Drive マウント完了")
        else:
            print("📂 Google Drive は既にマウントされています。")
    except Exception as e:
        print("⚠️ Google Driveマウントをスキップ (ローカル保存で継続):", e)

# 2. スクリプト群の自己展開
REPO_DIR = "/content/ColabTool/GamingPC"
SCRIPTS_DIR = os.path.join(REPO_DIR, "scripts")
os.makedirs(SCRIPTS_DIR, exist_ok=True)

EMBEDDED_FILES = {
    "scripts/setup_environment.sh": "#!/bin/bash\n# ==============================================================================\n# setup_environment.sh - Gaming PC on Google Colab (L4 GPU High-Memory) Setup\n# ==============================================================================\nset -e\n\nexport DEBIAN_FRONTEND=noninteractive\nREPO_DIR=\"$(cd \"$(dirname \"${BASH_SOURCE[0]}\")/..\" && pwd)\"\n\necho \"==================================================\"\necho \"  \ud83c\udfae Gaming PC \u4eee\u60f3\u74b0\u5883 \u81ea\u52d5\u30bb\u30c3\u30c8\u30a2\u30c3\u30d7\u958b\u59cb\"\necho \"  \u52d5\u4f5c\u74b0\u5883: Colab L4 GPU (24GB VRAM) / High-RAM\"\necho \"==================================================\"\n\n# 1. \u30bf\u30a4\u30e0\u30be\u30fc\u30f3 (Asia/Tokyo) & \u65e5\u672c\u8a9e\u74b0\u5883 (ja_JP.UTF-8) \u306e\u8a2d\u5b9a\necho \">> [1/9] \u65e5\u672c\u6a19\u6e96\u6642 (Asia/Tokyo) & \u65e5\u672c\u8a9e\u30ed\u30b1\u30fc\u30eb\u306e\u30bb\u30c3\u30c8\u30a2\u30c3\u30d7...\"\nln -sf /usr/share/zoneinfo/Asia/Tokyo /etc/localtime\necho \"Asia/Tokyo\" > /etc/timezone\n\ndpkg --add-architecture i386 || true\napt-get update -qq\n\napt-get install -y -qq locales fonts-noto-cjk fonts-noto-color-emoji fonts-ipafont-gothic > /dev/null 2>&1 || true\nlocale-gen ja_JP.UTF-8 > /dev/null 2>&1 || true\nupdate-locale LANG=ja_JP.UTF-8 > /dev/null 2>&1 || true\nexport LANG=ja_JP.UTF-8\nexport LC_ALL=ja_JP.UTF-8\nexport TZ=Asia/Tokyo\n\n# 2. Xfce4 \u30c7\u30b9\u30af\u30c8\u30c3\u30d7\u74b0\u5883, \u97f3\u58f0 (PulseAudio), OpenGL/Vulkan, VNC, noVNC\necho \">> [2/9] Xfce4 \u30c7\u30b9\u30af\u30c8\u30c3\u30d7\u30fb\u97f3\u58f0\u30fbOpenGL/Vulkan\u30fb\u753b\u9762\u63cf\u753b\u30c4\u30fc\u30eb\u306e\u30a4\u30f3\u30b9\u30c8\u30fc\u30eb...\"\napt-get install -y -qq \\\n    xfce4 xfce4-goodies xfce4-terminal dbus-x11 x11-xserver-utils x11-utils xterm \\\n    xvfb x11vnc websockify novnc \\\n    pulseaudio pulseaudio-utils alsa-utils pavucontrol \\\n    libgl1-mesa-glx mesa-utils libvulkan1 vulkan-tools \\\n    ffmpeg jq curl wget git zstd htop \\\n    python3-dbus python3-gi libdbus-1-3 libdbus-1-dev libglib2.0-dev python3-dev pkg-config \\\n    openjdk-21-jdk-headless > /dev/null 2>&1\n\n# Python websockify & D-Bus \u30d0\u30a4\u30f3\u30c7\u30a3\u30f3\u30b0\u306e\u78ba\u5b9f\u306a\u30b3\u30f3\u30d1\u30a4\u30eb\u30fb\u5c0e\u5165\npip install -q websockify > /dev/null 2>&1 || true\npython3 -m pip install --ignore-installed --no-cache-dir dbus-python > /dev/null 2>&1 || true\nif which python3.10 >/dev/null 2>&1 && python3.10 -c \"import dbus\" >/dev/null 2>&1; then\n    sed -i '1s|.*|#!/usr/bin/python3.10|' /opt/google/chrome-remote-desktop/chrome-remote-desktop 2>/dev/null || true\nfi\n\n# 3. Google Chrome & Chrome Remote Desktop \u306e\u30a4\u30f3\u30b9\u30c8\u30fc\u30eb\necho \">> [3/9] Google Chrome & Chrome Remote Desktop \u306e\u78ba\u8a8d & \u30a4\u30f3\u30b9\u30c8\u30fc\u30eb...\"\nif ! which google-chrome > /dev/null 2>&1 && ! which google-chrome-stable > /dev/null 2>&1; then\n    wget -q -O /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb\n    apt-get install -y -qq /tmp/chrome.deb > /dev/null 2>&1 || apt-get install -f -y -qq > /dev/null 2>&1\n    rm -f /tmp/chrome.deb\nfi\n\n# Chrome root\u5b9f\u884c\u7528\u30e9\u30c3\u30d1\u30fc\ncat << 'EOF' > /usr/local/bin/google-chrome-colab\n#!/bin/bash\nexec /usr/bin/google-chrome-stable --no-sandbox --disable-dev-shm-usage --disable-gpu-sandbox \"$@\"\nEOF\nchmod +x /usr/local/bin/google-chrome-colab\n\nif ! dpkg -s chrome-remote-desktop > /dev/null 2>&1; then\n    wget -q -O /tmp/crd.deb https://dl.google.com/linux/direct/chrome-remote-desktop_current_amd64.deb\n    apt-get install -y -qq /tmp/crd.deb > /dev/null 2>&1 || apt-get install -f -y -qq > /dev/null 2>&1\n    rm -f /tmp/crd.deb\nfi\n\n# Colab/Docker\u74b0\u5883\u7528 systemctl \u30e2\u30c3\u30af (CRD\u306estart-host\u304csystemd\u4e0d\u5728\u3067\u30af\u30e9\u30c3\u30b7\u30e5\u3059\u308b\u306e\u3092\u9632\u3050)\ncat << 'EOF' > /usr/local/bin/systemctl\n#!/bin/bash\nif [ \"$1\" = \"is-active\" ]; then\n    echo \"active\"\nfi\nexit 0\nEOF\nchmod +x /usr/local/bin/systemctl\n\n# 4. \u4e00\u822c\u30e6\u30fc\u30b6\u30fc (colabuser) \u306e\u4f5c\u6210 & \u6a29\u9650\u4ed8\u4e0e (CRD\u30fbSteam \u306e root \u5b9f\u884c\u5236\u9650\u5bfe\u7b56)\necho \">> [4/9] \u4e00\u822c\u30e6\u30fc\u30b6\u30fc (colabuser) \u306e\u69cb\u6210 & \u6a29\u9650\u8a2d\u5b9a...\"\nif ! id -u colabuser >/dev/null 2>&1; then\n    useradd -m -s /bin/bash -G sudo,chrome-remote-desktop,video,audio colabuser\n    echo \"colabuser:123456\" | chpasswd\n    echo \"colabuser ALL=(ALL) NOPASSWD:ALL\" > /etc/sudoers.d/colabuser\nfi\nusermod -aG video,audio,render colabuser 2>/dev/null || true\n\n# X11 \u753b\u9762\u3078\u306e\u30ed\u30fc\u30ab\u30eb\u30e6\u30fc\u30b6\u30fc\u30a2\u30af\u30bb\u30b9\u8a31\u53ef\nxhost +local: > /dev/null 2>&1 || xhost + > /dev/null 2>&1 || true\n\nmkdir -p /root/.config/chrome-remote-desktop /home/colabuser/.config/chrome-remote-desktop\necho \"exec /etc/X11/Xsession /usr/bin/xfce4-session\" > /root/.chrome-remote-desktop-session\necho \"exec /etc/X11/Xsession /usr/bin/xfce4-session\" > /home/colabuser/.chrome-remote-desktop-session\nchmod +x /root/.chrome-remote-desktop-session /home/colabuser/.chrome-remote-desktop-session\nchown -R colabuser:colabuser /home/colabuser\n\n# 5. Steam \u306e\u516c\u5f0f\u30a4\u30f3\u30b9\u30c8\u30fc\u30e9\u30fc\u5c0e\u5165 & root \u56de\u907f\u30e9\u30c3\u30d1\u30fc\u8a2d\u5b9a\necho \">> [5/9] Steam \u306e\u78ba\u8a8d & \u30a4\u30f3\u30b9\u30c8\u30fc\u30eb...\"\nif ! which steam > /dev/null 2>&1 && [ ! -f /usr/games/steam ]; then\n    wget -q -O /tmp/steam.deb \"https://cdn.cloudflare.steamstatic.com/client/installer/steam.deb\" || true\n    if [ -f /tmp/steam.deb ]; then\n        dpkg -i /tmp/steam.deb > /dev/null 2>&1 || apt-get install -f -y -qq > /dev/null 2>&1 || true\n        rm -f /tmp/steam.deb\n    fi\nfi\n\n# \u4e00\u822c\u30e6\u30fc\u30b6\u30fc colabuser \u3068\u3057\u3066 Steam \u3092\u8d77\u52d5\u3059\u308b\u30e9\u30c3\u30d1\u30fc\ncat << 'EOF' > /usr/local/bin/steam-colab\n#!/bin/bash\nxhost +local: > /dev/null 2>&1 || xhost + > /dev/null 2>&1 || true\nexport DISPLAY=:1\nexec sudo -u colabuser DISPLAY=:1 HOME=/home/colabuser /usr/games/steam \"$@\"\nEOF\nchmod +x /usr/local/bin/steam-colab\n\n# 6. Discord \u306e\u30a4\u30f3\u30b9\u30c8\u30fc\u30eb & root \u8d77\u52d5\u30e9\u30c3\u30d1\u30fc\u8a2d\u5b9a\necho \">> [6/9] Discord \u306e\u78ba\u8a8d & \u30a4\u30f3\u30b9\u30c8\u30fc\u30eb...\"\nif ! which discord > /dev/null 2>&1; then\n    wget -q -O /tmp/discord.deb \"https://discord.com/api/download?platform=linux&format=deb\" || true\n    if [ -f /tmp/discord.deb ]; then\n        apt-get install -y -qq /tmp/discord.deb > /dev/null 2>&1 || apt-get install -f -y -qq > /dev/null 2>&1 || true\n        rm -f /tmp/discord.deb\n    fi\nfi\ncat << 'EOF' > /usr/local/bin/discord-colab\n#!/bin/bash\nexec /usr/bin/discord --no-sandbox \"$@\"\nEOF\nchmod +x /usr/local/bin/discord-colab\n\n# 7. CurseForge & Prism Launcher (Minecraft\u30af\u30e9\u30a4\u30a2\u30f3\u30c8) & OBS Studio\necho \">> [7/9] CurseForge / Prism Launcher / OBS Studio \u306e\u30bb\u30c3\u30c8\u30a2\u30c3\u30d7...\"\nmkdir -p /usr/local/bin /opt/curseforge /opt/prismlauncher\n\n# CurseForge AppImage\nif [ ! -f /opt/curseforge/curseforge.AppImage ]; then\n    curl -sSL -o /opt/curseforge/curseforge.AppImage \"https://curseforge.overwolf.com/downloads/curseforge-latest-linux.AppImage\" || true\n    chmod +x /opt/curseforge/curseforge.AppImage 2>/dev/null || true\nfi\n\n# Prism Launcher (\u9ad8\u6027\u80fdMinecraft\u30e9\u30f3\u30c1\u30e3\u30fc)\nif [ ! -f /opt/prismlauncher/prismlauncher.AppImage ]; then\n    curl -sSL -o /opt/prismlauncher/prismlauncher.AppImage \"https://github.com/PrismLauncher/PrismLauncher/releases/download/8.4/PrismLauncher-Linux-8.4-x86_64.AppImage\" || \\\n    curl -sSL -o /opt/prismlauncher/prismlauncher.AppImage \"https://github.com/PrismLauncher/PrismLauncher/releases/latest/download/PrismLauncher-Linux-x86_64.AppImage\" || true\n    chmod +x /opt/prismlauncher/prismlauncher.AppImage 2>/dev/null || true\nfi\n\ncat << 'EOF' > /usr/local/bin/prismlauncher\n#!/bin/bash\nexec /opt/prismlauncher/prismlauncher.AppImage --appimage-extract-and-run \"$@\"\nEOF\nchmod +x /usr/local/bin/prismlauncher\n\n# OBS Studio\nif ! which obs > /dev/null 2>&1; then\n    apt-get install -y -qq obs-studio > /dev/null 2>&1 || true\nfi\n\n# 8. \u30c8\u30f3\u30cd\u30eb\u30c4\u30fc\u30eb (Cloudflared & playit.gg)\necho \">> [8/9] \u30c8\u30f3\u30cd\u30eb\u30c4\u30fc\u30eb (Cloudflared & playit.gg) \u306e\u30bb\u30c3\u30c8\u30a2\u30c3\u30d7...\"\nif [ ! -f /usr/local/bin/cloudflared ]; then\n    curl -sSL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64\n    chmod +x /usr/local/bin/cloudflared\nfi\n\nif [ ! -f /usr/local/bin/playit ]; then\n    curl -sSL -o /usr/local/bin/playit https://github.com/playit-cloud/playit-agent/releases/download/v0.15.26/playit-linux-amd64\n    chmod +x /usr/local/bin/playit\nfi\n\n# 9. \u30c7\u30b9\u30af\u30c8\u30c3\u30d7\u74b0\u5883\u30fb\u30b7\u30e7\u30fc\u30c8\u30ab\u30c3\u30c8\u30fb\u30d7\u30ed\u30d5\u30a1\u30a4\u30eb\u6574\u5099\necho \">> [9/9] \u65e5\u672c\u8a9e\u30c7\u30b9\u30af\u30c8\u30c3\u30d7\u30b7\u30e7\u30fc\u30c8\u30ab\u30c3\u30c8 & \u6a29\u9650\u767b\u9332...\"\nDESKTOP_DIR=\"/root/Desktop\"\nmkdir -p \"$DESKTOP_DIR\" /content/chrome_profiles/profile1 /content/chrome_profiles/profile2 /content/chrome_profiles/profile3 /content/recordings\n\n# Chrome Profile 1\u301c3\nfor i in 1 2 3; do\ncat << EOF > \"$DESKTOP_DIR/Chrome_Profile_$i.desktop\"\n[Desktop Entry]\nVersion=1.0\nType=Application\nName=Chrome (Profile $i)\nComment=Chrome Profile $i\nExec=/usr/local/bin/google-chrome-colab --user-data-dir=/content/chrome_profiles/profile$i\nIcon=google-chrome\nTerminal=false\nCategories=Network;WebBrowser;\nEOF\ndone\n\n# Steam\ncat << 'EOF' > \"$DESKTOP_DIR/Steam.desktop\"\n[Desktop Entry]\nVersion=1.0\nType=Application\nName=Steam (Hoi4 / Chill with You)\nComment=Play games on Steam\nExec=/usr/local/bin/steam-colab\nIcon=steam\nTerminal=false\nCategories=Game;\nEOF\n\n# Minecraft (Prism Launcher)\ncat << 'EOF' > \"$DESKTOP_DIR/Minecraft.desktop\"\n[Desktop Entry]\nVersion=1.0\nType=Application\nName=Minecraft (Prism Launcher)\nComment=Prism Launcher for Minecraft & Modpacks\nExec=/usr/local/bin/prismlauncher\nIcon=system-run\nTerminal=false\nCategories=Game;\nEOF\n\n# CurseForge\ncat << 'EOF' > \"$DESKTOP_DIR/CurseForge.desktop\"\n[Desktop Entry]\nVersion=1.0\nType=Application\nName=CurseForge (Mod\u7ba1\u7406)\nComment=CurseForge Mod Manager\nExec=/opt/curseforge/curseforge.AppImage --no-sandbox --appimage-extract-and-run\nIcon=applications-games\nTerminal=false\nCategories=Game;\nEOF\n\n# Xbox Cloud Gaming\ncat << 'EOF' > \"$DESKTOP_DIR/XboxCloud.desktop\"\n[Desktop Entry]\nVersion=1.0\nType=Application\nName=Xbox Cloud Gaming\nComment=Xbox Cloud Gaming on Browser\nExec=/usr/local/bin/google-chrome-colab --app=\"https://www.xbox.com/play\"\nIcon=input-gaming\nTerminal=false\nCategories=Game;\nEOF\n\n# Roblox\ncat << 'EOF' > \"$DESKTOP_DIR/Roblox.desktop\"\n[Desktop Entry]\nVersion=1.0\nType=Application\nName=Roblox\nComment=Roblox Gaming\nExec=/usr/local/bin/google-chrome-colab --app=\"https://www.roblox.com/home\"\nIcon=applications-games\nTerminal=false\nCategories=Game;\nEOF\n\n# ChillwithYou\ncat << 'EOF' > \"$DESKTOP_DIR/ChillwithYou.desktop\"\n[Desktop Entry]\nVersion=1.0\nType=Application\nName=ChillwithYou\nComment=Lo-Fi Audio & Relax Companion\nExec=/usr/local/bin/google-chrome-colab --app=\"https://chillhop.com/live/chillhop-radio/\"\nIcon=audio-headphones\nTerminal=false\nCategories=AudioVideo;Audio;\nEOF\n\n# Discord\ncat << 'EOF' > \"$DESKTOP_DIR/Discord.desktop\"\n[Desktop Entry]\nVersion=1.0\nType=Application\nName=Discord\nComment=All-in-one voice and text chat\nExec=/usr/local/bin/discord-colab\nIcon=discord\nTerminal=false\nCategories=Network;InstantMessaging;\nEOF\n\n# OBS Studio\ncat << 'EOF' > \"$DESKTOP_DIR/OBS.desktop\"\n[Desktop Entry]\nVersion=1.0\nType=Application\nName=OBS Studio (\u753b\u9762\u53ce\u9332\u30fb\u914d\u4fe1)\nComment=Screen Recording & Streaming\nExec=obs\nIcon=obs\nTerminal=false\nCategories=AudioVideo;Recorder;\nEOF\n\n# 1-Click \u753b\u9762\u53ce\u9332 (NVENC 60fps) \u30b7\u30e7\u30fc\u30c8\u30ab\u30c3\u30c8\ncat << 'EOF' > \"$DESKTOP_DIR/ScreenRecorder_Toggle.desktop\"\n[Desktop Entry]\nVersion=1.0\nType=Application\nName=\u753b\u9762\u53ce\u9332 (\u958b\u59cb/\u505c\u6b62)\nComment=Toggle Hardware Accelerated Screen Recording\nExec=/bin/bash -c \"/content/ColabTool/GamingPC/scripts/record_screen.sh toggle\"\nIcon=media-record\nTerminal=true\nCategories=AudioVideo;Recorder;\nEOF\n\n# Minecraft \u30b5\u30fc\u30d0\u30fc \u30ed\u30b0\u76e3\u8996\u30b3\u30f3\u30bd\u30fc\u30eb\ncat << 'EOF' > \"$DESKTOP_DIR/Minecraft_Server_Logs.desktop\"\n[Desktop Entry]\nVersion=1.0\nType=Application\nName=Minecraft \u30b5\u30fc\u30d0\u30fc \u30ed\u30b0\u76e3\u8996\nComment=View background Minecraft server logs\nExec=xterm -title \"Minecraft Server Logs\" -hold -e \"tail -n 100 -f /content/minecraft_server/logs/latest.log\"\nIcon=utilities-terminal\nTerminal=false\nCategories=System;\nEOF\n\n# \u97f3\u58f0\u30df\u30ad\u30b5\u30fc (pavucontrol) \u30b7\u30e7\u30fc\u30c8\u30ab\u30c3\u30c8\ncat << 'EOF' > \"$DESKTOP_DIR/Volume_Control.desktop\"\n[Desktop Entry]\nVersion=1.0\nType=Application\nName=\u97f3\u91cf\u30fb\u30b5\u30a6\u30f3\u30c9\u8a2d\u5b9a\nComment=PulseAudio Volume Control\nExec=pavucontrol\nIcon=audio-volume-high\nTerminal=false\nCategories=AudioVideo;\nEOF\n\n# \u5168\u30b7\u30e7\u30fc\u30c8\u30ab\u30c3\u30c8\u3092\u5b9f\u884c\u53ef\u80fd & \u4fe1\u983c\u6e08\u307f\u3068\u3057\u3066\u4e00\u62ec\u767b\u9332\nchmod +x \"$DESKTOP_DIR\"/*.desktop 2>/dev/null || true\nfor f in \"$DESKTOP_DIR\"/*.desktop; do\n    gio set \"$f\" metadata::trusted yes 2>/dev/null || true\ndone\n\nmkdir -p /home/colabuser/Desktop\ncp -rf \"$DESKTOP_DIR\"/* /home/colabuser/Desktop/ 2>/dev/null || true\nchmod +x /home/colabuser/Desktop/*.desktop 2>/dev/null || true\nfor f in /home/colabuser/Desktop/*.desktop; do\n    gio set \"$f\" metadata::trusted yes 2>/dev/null || true\ndone\nchown -R colabuser:colabuser /home/colabuser\n\n# Xfce4 \u30c7\u30d5\u30a9\u30eb\u30c8\u30d1\u30cd\u30eb\u8a2d\u5b9a\uff08\u521d\u56de\u8d77\u52d5\u306e\u9ed2\u753b\u9762\u30fb\u30c0\u30a4\u30a2\u30ed\u30b0\u9632\u6b62\uff09\nmkdir -p /root/.config/xfce4/xfconf/xfce-perchannel-xml /home/colabuser/.config/xfce4/xfconf/xfce-perchannel-xml\ncp -r /etc/xdg/xfce4/panel/default.xml /root/.config/xfce4/xfconf/xfce-perchannel-xml/xfce4-panel.xml 2>/dev/null || true\ncp -r /etc/xdg/xfce4/panel/default.xml /home/colabuser/.config/xfce4/xfconf/xfce-perchannel-xml/xfce4-panel.xml 2>/dev/null || true\nchown -R colabuser:colabuser /home/colabuser/.config\n\necho \"\u2705 [\u5b8c\u4e86] Gaming PC \u4eee\u60f3\u74b0\u5883\u306e\u30bb\u30c3\u30c8\u30a2\u30c3\u30d7\u304c\u6b63\u5e38\u306b\u5b8c\u4e86\u3057\u307e\u3057\u305f\u3002\"\n",
    "scripts/start_desktop.sh": "#!/bin/bash\n# ==============================================================================\n# start_desktop.sh - Start Xfce4, Virtual Audio, VNC, noVNC & Ultra-Sync Audio\n# ==============================================================================\nset -e\n\nCRD_CODE=\"$1\"\nCRD_PIN=\"${2:-123456}\"\nSCREEN_RES=\"${3:-1680x1050}\" # MacBook \u9ec4\u91d1\u6bd4 16:10 \u30c7\u30d5\u30a9\u30eb\u30c8\n\necho \"==================================================\"\necho \"  \ud83d\udda5\ufe0f \u30c7\u30b9\u30af\u30c8\u30c3\u30d7\u74b0\u5883 & \u30ea\u30a2\u30eb\u30bf\u30a4\u30e0\u63a5\u7d9a\u306e\u8d77\u52d5\"\necho \"  \u753b\u9762\u89e3\u50cf\u5ea6: ${SCREEN_RES} (MacBook 16:10 \u6700\u9069\u5316)\"\necho \"==================================================\"\n\n# 1. \u4eee\u60f3\u30aa\u30fc\u30c7\u30a3\u30aa (PulseAudio) \u306e\u8d77\u52d5\necho \">> [1/5] \u4eee\u60f3\u30aa\u30fc\u30c7\u30a3\u30aa (PulseAudio) \u3092\u521d\u671f\u5316\u4e2d...\"\npulseaudio -D --exit-idle-time=-1 2>/dev/null || true\npactl load-module module-null-sink sink_name=VirtualSink sink_properties=device.description=VirtualSink 2>/dev/null || true\npactl set-default-sink VirtualSink 2>/dev/null || true\n\n# 2. Xvfb \u4eee\u60f3\u30c7\u30a3\u30b9\u30d7\u30ec\u30a4 & Xfce4 \u306e\u8d77\u52d5\necho \">> [2/5] Xvfb \u4eee\u60f3\u30c7\u30a3\u30b9\u30d7\u30ec\u30a4 (${SCREEN_RES} 24bit) & Xfce4 \u3092\u8d77\u52d5\u4e2d...\"\nexport DISPLAY=:1\nexport LANG=ja_JP.UTF-8\nexport LC_ALL=ja_JP.UTF-8\nexport TZ=Asia/Tokyo\n\nif ! pgrep -f \"Xvfb :1\" > /dev/null 2>&1; then\n    nohup Xvfb :1 -screen 0 \"${SCREEN_RES}x24\" +extension GLX +extension RENDER -noreset > /content/xvfb.log 2>&1 &\n    sleep 2\nfi\n\n# Xfce4 \u30d1\u30cd\u30eb\u521d\u671f\u8a2d\u5b9a\u306e\u914d\u7f6e\uff08\u521d\u56de\u9ed2\u753b\u9762\u9632\u6b62\uff09\nmkdir -p /root/.config/xfce4/xfconf/xfce-perchannel-xml\nif [ ! -f /root/.config/xfce4/xfconf/xfce-perchannel-xml/xfce4-panel.xml ]; then\n    cp -r /etc/xdg/xfce4/panel/default.xml /root/.config/xfce4/xfconf/xfce-perchannel-xml/xfce4-panel.xml 2>/dev/null || true\nfi\n\nif ! pgrep -f \"xfce4-session\" > /dev/null 2>&1; then\n    nohup dbus-launch --exit-with-session startxfce4 > /content/xfce.log 2>&1 &\n    sleep 2\nfi\n\n# 3. \u8d85\u4f4e\u9045\u5ef6 PCM \u97f3\u58f0\u914d\u4fe1\u30b5\u30fc\u30d0\u30fc (\u30dd\u30fc\u30c8 8000)\necho \">> [3/5] \u8d85\u4f4e\u9045\u5ef6\u30ea\u30a2\u30eb\u30bf\u30a4\u30e0\u97f3\u58f0\u914d\u4fe1\u30c7\u30fc\u30e2\u30f3 (\u30dd\u30fc\u30c8 8000) \u3092\u8d77\u52d5...\"\nfuser -k 8000/tcp 2>/dev/null || true\n\ncat << 'EOF' > /content/audio_stream.py\nimport subprocess\nfrom http.server import HTTPServer, BaseHTTPRequestHandler\n\nclass UltraLowLatencyHandler(BaseHTTPRequestHandler):\n    def do_GET(self):\n        self.send_response(200)\n        self.send_header('Content-Type', 'application/octet-stream')\n        self.send_header('Cache-Control', 'no-cache, no-store, must-revalidate')\n        self.send_header('Pragma', 'no-cache')\n        self.send_header('Access-Control-Allow-Origin', '*')\n        self.send_header('Connection', 'close')\n        self.end_headers()\n        \n        cmd = [\n            \"ffmpeg\", \"-f\", \"pulse\", \"-i\", \"VirtualSink.monitor\",\n            \"-ac\", \"2\", \"-ar\", \"44100\", \"-f\", \"s16le\",\n            \"-bufsize\", \"1024\", \"-fflags\", \"nobuffer\", \"-flush_packets\", \"1\", \"-\"\n        ]\n        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, bufsize=512)\n        try:\n            while True:\n                chunk = proc.stdout.read(2048)\n                if not chunk:\n                    break\n                self.wfile.write(chunk)\n        except Exception:\n            pass\n        finally:\n            proc.kill()\n\nhttpd = HTTPServer(('0.0.0.0', 8000), UltraLowLatencyHandler)\nhttpd.serve_forever()\nEOF\nnohup python3 /content/audio_stream.py > /dev/null 2>&1 &\n\n# 4. x11vnc & noVNC \u306e\u8d77\u52d5\necho \">> [4/5] x11vnc & noVNC \u30b5\u30fc\u30d0\u30fc\u3092\u30d0\u30c3\u30af\u30b0\u30e9\u30a6\u30f3\u30c9\u8d77\u52d5\u4e2d...\"\nfuser -k 5901/tcp 6080/tcp 2>/dev/null || true\nsleep 1\n\nnohup x11vnc -display :1 -nopw -listen 0.0.0.0 -xkb -ncache 10 -ncache_cr -forever -shared -rfbport 5901 > /content/x11vnc.log 2>&1 &\n\nNOVNC_DIR=\"/usr/share/novnc\"\n[ -d \"$NOVNC_DIR\" ] || NOVNC_DIR=\"/usr/local/share/novnc\"\n[ -f \"$NOVNC_DIR/index.html\" ] && [ ! -f \"$NOVNC_DIR/vnc.html\" ] && ln -sf \"$NOVNC_DIR/index.html\" \"$NOVNC_DIR/vnc.html\"\n\n# noVNC \u306e HTML \u306b Web Audio API \u30ea\u30a2\u30eb\u30bf\u30a4\u30e0\u97f3\u540c\u671f\u30a6\u30a3\u30b8\u30a7\u30c3\u30c8\u3092\u6ce8\u5165\npython3 - << 'PYEOF'\nimport os, re\nnovnc_dir = \"/usr/share/novnc\" if os.path.exists(\"/usr/share/novnc\") else \"/usr/local/share/novnc\"\ntarget_html = f\"{novnc_dir}/vnc.html\"\n\nif os.path.exists(target_html):\n    with open(target_html, \"r\", encoding=\"utf-8\") as f:\n        content = f.read()\n\n    audio_script = '''\n<!-- ANTIGRAVITY AUDIO INTEGRATION -->\n<div id=\"live-audio-bar\" style=\"position: fixed; top: 12px; right: 80px; z-index: 99999; background: rgba(15, 23, 42, 0.9); backdrop-filter: blur(8px); border: 1px solid rgba(56, 189, 248, 0.5); border-radius: 30px; padding: 6px 16px; display: flex; align-items: center; gap: 10px; color: #fff; font-family: -apple-system, BlinkMacSystemFont, sans-serif; font-size: 13px; box-shadow: 0 4px 15px rgba(0,0,0,0.5); user-select: none;\">\n  <span id=\"audio-toggle\" style=\"cursor: pointer; font-size: 16px;\" title=\"\u30af\u30ea\u30c3\u30af\u3067\u30df\u30e5\u30fc\u30c8/\u89e3\u9664\">\ud83d\udd0a</span>\n  <span style=\"font-weight: bold; color: #38bdf8; font-size: 11px; letter-spacing: 0.5px;\">LIVE SOUND</span>\n  <input id=\"audio-volume\" type=\"range\" min=\"0\" max=\"1\" step=\"0.05\" value=\"0.8\" style=\"width: 70px; cursor: pointer; accent-color: #38bdf8;\">\n</div>\n<script>\n(function() {\n  let audioCtx = null, gainNode = null, isRunning = false, nextStartTime = 0;\n  const toggleBtn = document.getElementById('audio-toggle');\n  const volumeSlider = document.getElementById('audio-volume');\n  let currentVolume = 0.8;\n\n  async function initWebAudio() {\n    if (isRunning) return;\n    isRunning = true;\n    audioCtx = new (window.AudioContext || window.webkitAudioContext)({ latencyHint: 'interactive', sampleRate: 44100 });\n    gainNode = audioCtx.createGain();\n    gainNode.gain.value = currentVolume;\n    gainNode.connect(audioCtx.destination);\n    nextStartTime = audioCtx.currentTime;\n\n    const audioUrl = window.location.origin.replace(/^https?:\\\\/\\\\/6080-/, (m) => m.replace('6080-', '8000-')) + '/stream';\n\n    try {\n      const response = await fetch(audioUrl);\n      const reader = response.body.getReader();\n\n      while (isRunning) {\n        const { done, value } = await reader.read();\n        if (done) break;\n\n        const int16 = new Int16Array(value.buffer, value.byteOffset, Math.floor(value.byteLength / 2));\n        const numSamples = Math.floor(int16.length / 2);\n        if (numSamples === 0) continue;\n\n        const buffer = audioCtx.createBuffer(2, numSamples, 44100);\n        const left = buffer.getChannelData(0);\n        const right = buffer.getChannelData(1);\n\n        for (let i = 0; i < numSamples; i++) {\n          left[i] = int16[i * 2] / 32768;\n          right[i] = int16[i * 2 + 1] / 32768;\n        }\n\n        const source = audioCtx.createBufferSource();\n        source.buffer = buffer;\n        source.connect(gainNode);\n\n        if (nextStartTime < audioCtx.currentTime - 0.05) {\n          nextStartTime = audioCtx.currentTime;\n        }\n        source.start(nextStartTime);\n        nextStartTime += buffer.duration;\n      }\n    } catch (e) {\n      isRunning = false;\n    }\n  }\n\n  document.addEventListener('click', () => {\n    if (!audioCtx || audioCtx.state === 'suspended') {\n      if (audioCtx) audioCtx.resume();\n      initWebAudio();\n    }\n  }, { once: true });\n\n  toggleBtn.addEventListener('click', (e) => {\n    e.stopPropagation();\n    if (!gainNode) return;\n    if (gainNode.gain.value > 0) {\n      gainNode.gain.value = 0;\n      toggleBtn.textContent = '\ud83d\udd07';\n    } else {\n      gainNode.gain.value = currentVolume;\n      toggleBtn.textContent = '\ud83d\udd0a';\n    }\n  });\n\n  volumeSlider.addEventListener('input', (e) => {\n    e.stopPropagation();\n    currentVolume = parseFloat(e.target.value);\n    if (gainNode) gainNode.gain.value = currentVolume;\n    toggleBtn.textContent = currentVolume === 0 ? '\ud83d\udd07' : '\ud83d\udd0a';\n  });\n})();\n</script>\n<!-- END ANTIGRAVITY AUDIO INTEGRATION -->\n'''\n    content = re.sub(r'<!-- ANTIGRAVITY AUDIO INTEGRATION -->.*?<!-- END ANTIGRAVITY AUDIO INTEGRATION -->', '', content, flags=re.DOTALL)\n    content = content.replace(\"</body>\", f\"{audio_script}\\n</body>\")\n    with open(target_html, \"w\", encoding=\"utf-8\") as f:\n        f.write(content)\nPYEOF\n\nnohup websockify --web=\"$NOVNC_DIR\" 6080 127.0.0.1:5901 > /content/novnc.log 2>&1 &\nsleep 2\n\n# Cloudflared \u9ad8\u901f\u30c8\u30f3\u30cd\u30eb\u306e\u8d77\u52d5 (\u5916\u90e8\u30d0\u30c3\u30af\u30a2\u30c3\u30d7\u7528)\npkill -f \"cloudflared tunnel\" 2>/dev/null || true\nrm -f /content/cloudflared.log\nnohup cloudflared tunnel --url http://127.0.0.1:6080 > /content/cloudflared.log 2>&1 &\n\nNOVNC_URL=\"\"\nfor i in {1..15}; do\n    NOVNC_URL=$(grep -oE \"https://[a-zA-Z0-9.-]+\\.trycloudflare\\.com\" /content/cloudflared.log 2>/dev/null | head -n 1 || true)\n    if [ -n \"$NOVNC_URL\" ]; then\n        break\n    fi\n    sleep 1\ndone\n\n# 5. Chrome Remote Desktop (CRD) \u306e\u8a2d\u5b9a\uff08\u6307\u5b9a\u3055\u308c\u3066\u3044\u308b\u5834\u5408\uff09\necho \">> [5/5] \u30ea\u30e2\u30fc\u30c8\u63a5\u7d9a\u65b9\u5f0f\u306e\u69cb\u6210...\"\nif [ -n \"$CRD_CODE\" ]; then\n    echo \"Chrome Remote Desktop \u3092\u69cb\u6210\u4e2d...\"\n    cat << 'EOF' > /usr/local/bin/systemctl\n#!/bin/bash\nif [ \"$1\" = \"is-active\" ]; then\n    echo \"active\"\nfi\nexit 0\nEOF\n    chmod +x /usr/local/bin/systemctl\n\n    if [[ \"$CRD_CODE\" == *\"start-host\"* ]]; then\n        CMD=\"$CRD_CODE\"\n        if [[ \"$CMD\" != *\"--pin=\"* ]]; then\n            CMD=\"$CMD --pin=$CRD_PIN\"\n        fi\n    else\n        CMD=\"DISPLAY= /opt/google/chrome-remote-desktop/start-host --code=\\\"$CRD_CODE\\\" --redirect-url=\\\"https://remotedesktop.google.com/_/oauthredirect\\\" --name=\\$(hostname) --pin=$CRD_PIN\"\n    fi\n\n    pkill -u colabuser -f chrome-remote-desktop 2>/dev/null || true\n    service dbus start > /dev/null 2>&1 || true\n    sudo -u colabuser bash -c \"$CMD\" > /content/crd_setup.log 2>&1 || true\n    nohup sudo -u colabuser /opt/google/chrome-remote-desktop/chrome-remote-desktop > /content/crd_run.log 2>&1 &\n    sleep 2\nfi\n\necho \"\"\necho \"==================================================\"\necho \"\ud83c\udf89 \u30c7\u30b9\u30af\u30c8\u30c3\u30d7\u74b0\u5883\u306e\u8d77\u52d5\u304c\u5b8c\u4e86\u3057\u307e\u3057\u305f\uff01\"\necho \"==================================================\"\nif [ -n \"$NOVNC_URL\" ]; then\n    echo \"\ud83c\udf10 \u3010Cloudflare \u9ad8\u901f\u30c8\u30f3\u30cd\u30eb URL\u3011:\"\n    echo \"\ud83d\udc49  ${NOVNC_URL}/vnc.html?autoconnect=true&resize=scale\"\n    echo \"--------------------------------------------------\"\nfi\necho \"\u2728 \u89e3\u50cf\u5ea6: ${SCREEN_RES} (MacBook 16:10)\"\necho \"\u2728 \u97f3\u58f0\u540c\u671f: Web Audio API (\u8d85\u4f4e\u9045\u5ef6\u30fb\u30ea\u30a2\u30eb\u30bf\u30a4\u30e0\u97f3\u540c\u671f)\"\necho \"==================================================\"\n",
    "scripts/start_mc_server.sh": "#!/bin/bash\n# ==============================================================================\n# start_mc_server.sh - Decoupled Background Minecraft Paper Server\n# (Runs independently of notebook cells; uses named pipe for commands)\n# ==============================================================================\nset -e\n\nMC_RAM_GB=\"${1:-8}\"\nSERVER_DIR=\"/content/minecraft_server\"\nMINECRAFT_VERSION=\"1.21.1\"\n\necho \"==================================================\"\necho \"  \u26cf\ufe0f \u72ec\u7acbMinecraft\u30b5\u30fc\u30d0\u30fc (PaperMC) \u8d77\u52d5\"\necho \"  \u5272\u308a\u5f53\u3066RAM: ${MC_RAM_GB}GB (Colab High-RAM\u6700\u9069\u5316)\"\necho \"==================================================\"\n\nmkdir -p \"$SERVER_DIR/logs\" \"$SERVER_DIR/plugins\"\ncd \"$SERVER_DIR\"\n\n# EULA \u540c\u610f\necho \"eula=true\" > eula.txt\n\n# server.properties \u6700\u9069\u5316\nif [ ! -f \"server.properties\" ]; then\ncat << 'EOF' > server.properties\nserver-port=25565\nonline-mode=false\ndifficulty=normal\ngamemode=survival\npvp=true\nview-distance=10\nsimulation-distance=8\nmax-players=20\nenable-command-block=true\nmotd=Colab Gaming PC Minecraft Server\nnetwork-compression-threshold=256\nsync-chunk-writes=false\nallow-flight=true\nspawn-protection=0\nEOF\nfi\n\n# PaperMC jar \u306e\u30c0\u30a6\u30f3\u30ed\u30fc\u30c9 (\u672a\u30c0\u30a6\u30f3\u30ed\u30fc\u30c9\u6642)\nif [ ! -f \"server.jar\" ]; then\n    echo \">> PaperMC ($MINECRAFT_VERSION) \u306e\u30c0\u30a6\u30f3\u30ed\u30fc\u30c9\u4e2d...\"\n    LATEST_BUILD=$(curl -s \"https://api.papermc.io/v2/projects/paper/versions/${MINECRAFT_VERSION}\" | jq '.builds[-1]' 2>/dev/null || true)\n    if [ \"$LATEST_BUILD\" != \"null\" ] && [ -n \"$LATEST_BUILD\" ]; then\n        DOWNLOAD_URL=\"https://api.papermc.io/v2/projects/paper/versions/${MINECRAFT_VERSION}/builds/${LATEST_BUILD}/downloads/paper-${MINECRAFT_VERSION}-${LATEST_BUILD}.jar\"\n        curl -sL -o server.jar \"$DOWNLOAD_URL\"\n    else\n        curl -sL -o server.jar \"https://api.purpurmc.org/v2/purpur/${MINECRAFT_VERSION}/latest/download\"\n    fi\nfi\n\n# \u65e2\u5b58\u30b5\u30fc\u30d0\u30fc\u30d7\u30ed\u30bb\u30b9\u306e\u78ba\u8a8d\nif pgrep -f \"server.jar\" > /dev/null 2>&1; then\n    echo \"\u2705 Minecraft\u30b5\u30fc\u30d0\u30fc\u306f\u65e2\u306b\u30d0\u30c3\u30af\u30b0\u30e9\u30a6\u30f3\u30c9\u3067\u7a3c\u50cd\u4e2d\u3067\u3059\u3002\"\nelse\n    echo \">> \u30b5\u30fc\u30d0\u30fc\u30d7\u30ed\u30bb\u30b9\u3092\u30d1\u30a4\u30d7\u4ed8\u304d\u3067\u30d0\u30c3\u30af\u30b0\u30e9\u30a6\u30f3\u30c9\u8d77\u52d5\u4e2d...\"\n    PIPE_PATH=\"$SERVER_DIR/server.pipe\"\n    rm -f \"$PIPE_PATH\"\n    mkfifo \"$PIPE_PATH\"\n\n    # \u30d0\u30c3\u30af\u30b0\u30e9\u30a6\u30f3\u30c9\u8d77\u52d5 (nohup + tail -f FIFO \u30d1\u30a4\u30d7)\n    nohup tail -f \"$PIPE_PATH\" | java -Xms2G -Xmx${MC_RAM_GB}G -XX:+UseG1GC -jar server.jar nogui > \"$SERVER_DIR/logs/latest.log\" 2>&1 &\n    MC_PID=$!\n    echo \"$MC_PID\" > /content/mc_server.pid\n\n    echo \"\u8d77\u52d5\u5f85\u6a5f\u4e2d (\u7d0415\u79d2)...\"\n    for i in {1..15}; do\n        if grep -q \"Done (\" \"$SERVER_DIR/logs/latest.log\" 2>/dev/null; then\n            break\n        fi\n        sleep 1\n        echo -n \".\"\n    done\n    echo \"\"\nfi\n\n# playit.gg \u30c8\u30f3\u30cd\u30eb\u306e\u8d77\u52d5 (\u30dd\u30fc\u30c825565\u306e\u5916\u90e8\u516c\u958b)\nif ! pgrep -f \"playit\" > /dev/null 2>&1; then\n    echo \">> playit.gg \u30c8\u30f3\u30cd\u30eb\u3092\u8d77\u52d5\u4e2d...\"\n    nohup playit --stdout > /content/playit.log 2>&1 &\n    sleep 3\nfi\n\nPLAYIT_URL=$(grep -oE \"([a-zA-Z0-9.-]+\\.ply\\.gg:[0-9]+|[a-zA-Z0-9.-]+\\.playit\\.plus)\" /content/playit.log 2>/dev/null | tail -n 1 || true)\nCLAIM_URL=$(grep -oE \"https://playit\\.gg/claim/[a-zA-Z0-9]+\" /content/playit.log 2>/dev/null | head -n 1 || true)\n\necho \"==================================================\"\necho \"\u2705 Minecraft\u30b5\u30fc\u30d0\u30fc\u306f\u30bb\u30eb\u304b\u3089\u72ec\u7acb\u3057\u3066\u30d0\u30c3\u30af\u30b0\u30e9\u30a6\u30f3\u30c9\u3067\u7a3c\u50cd\u3057\u3066\u3044\u307e\u3059\u3002\"\necho \"\u30ed\u30fc\u30ab\u30eb\u63a5\u7d9a: localhost:25565\"\nif [ -n \"$PLAYIT_URL\" ]; then\n    echo \"\u5916\u90e8\u63a5\u7d9a\u30a2\u30c9\u30ec\u30b9: $PLAYIT_URL\"\nelif [ -n \"$CLAIM_URL\" ]; then\n    echo \"playit \u30c8\u30f3\u30cd\u30eb\u5272\u308a\u5f53\u3066\u30ea\u30f3\u30af: $CLAIM_URL\"\nfi\necho \"==================================================\"\n",
    "scripts/record_screen.sh": "#!/bin/bash\n# ==============================================================================\n# record_screen.sh - Hardware-Accelerated (NVENC 60fps) Screen Recording\n# (Supports start, stop, status, toggle with PulseAudio sound)\n# ==============================================================================\nset -e\n\nACTION=\"${1:-status}\"\nPID_FILE=\"/content/recorder.pid\"\nRECORD_DIR=\"/content/recordings\"\nGDRIVE_REC=\"/content/drive/MyDrive/GamingPC/recordings\"\n\nmkdir -p \"$RECORD_DIR\"\n\nstart_recording() {\n    if [ -f \"$PID_FILE\" ] && kill -0 \"$(cat \"$PID_FILE\")\" 2>/dev/null; then\n        echo \"\u26a0\ufe0f \u65e2\u306b\u753b\u9762\u53ce\u9332\u304c\u5b9f\u884c\u4e2d\u3067\u3059 (PID: $(cat \"$PID_FILE\"))\"\n        return 0\n    fi\n\n    TIMESTAMP=$(date +\"%Y%m%d_%H%M%S\")\n    OUT_FILE=\"$RECORD_DIR/Screen_${TIMESTAMP}.mp4\"\n    echo \">> \u753b\u9762\u53ce\u9332\u3092\u958b\u59cb\u3057\u307e\u3059: $OUT_FILE\"\n\n    # PulseAudio \u30e2\u30cb\u30bf\u30fc\u30bd\u30fc\u30b9\u53d6\u5f97\n    AUDIO_SRC=$(pactl list short sources 2>/dev/null | grep -E \"VirtualSink|monitor\" | head -n 1 | awk '{print $2}' || true)\n    if [ -z \"$AUDIO_SRC\" ]; then\n        AUDIO_SRC=\"default\"\n    fi\n\n    # L4 GPU (NVENC) \u30cf\u30fc\u30c9\u30a6\u30a7\u30a2\u30a8\u30f3\u30b3\u30fc\u30c9\n    # \u97f3\u58f0\u3068\u6620\u50cf\u3092\u540c\u671f\u3057\u3066 1920x1080 60fps \u53ce\u9332\n    if ffmpeg -encoders 2>&1 | grep -q \"h264_nvenc\"; then\n        echo \">> NVIDIA NVENC \u30cf\u30fc\u30c9\u30a6\u30a7\u30a2\u30a2\u30af\u30bb\u30e9\u30ec\u30fc\u30b7\u30e7\u30f3\u3092\u4f7f\u7528\u3057\u307e\u3059 (L4 GPU)\"\n        nohup ffmpeg -y \\\n            -f x11grab -draw_mouse 1 -framerate 60 -video_size 1920x1080 -i :1.0 \\\n            -f pulse -i \"$AUDIO_SRC\" \\\n            -c:v h264_nvenc -preset p4 -tune ll -b:v 8M -maxrate 12M -bufsize 16M \\\n            -c:a aac -b:a 192k \\\n            -pix_fmt yuv420p \"$OUT_FILE\" > /content/ffmpeg_record.log 2>&1 &\n    else\n        echo \">> \u30bd\u30d5\u30c8\u30a6\u30a7\u30a2\u30a8\u30f3\u30b3\u30fc\u30c9 (libx264) \u3092\u4f7f\u7528\u3057\u307e\u3059\"\n        nohup ffmpeg -y \\\n            -f x11grab -draw_mouse 1 -framerate 30 -video_size 1920x1080 -i :1.0 \\\n            -f pulse -i \"$AUDIO_SRC\" \\\n            -c:v libx264 -preset veryfast -crf 23 \\\n            -c:a aac -b:a 192k \\\n            -pix_fmt yuv420p \"$OUT_FILE\" > /content/ffmpeg_record.log 2>&1 &\n    fi\n\n    REC_PID=$!\n    echo \"$REC_PID\" > \"$PID_FILE\"\n    echo \"$OUT_FILE\" > /content/recorder_current_file.txt\n    sleep 2\n\n    if kill -0 \"$REC_PID\" 2>/dev/null; then\n        echo \"\u2705 \u753b\u9762\u53ce\u9332\u3092\u958b\u59cb\u3057\u307e\u3057\u305f (PID: $REC_PID)\"\n    else\n        echo \"\u26a0\ufe0f \u53ce\u9332\u958b\u59cb\u306b\u5931\u6557\u3057\u307e\u3057\u305f\u3002\u30ed\u30b0\u3092\u78ba\u8a8d\u3057\u3066\u304f\u3060\u3055\u3044:\"\n        tail -n 15 /content/ffmpeg_record.log 2>/dev/null || true\n    fi\n}\n\nstop_recording() {\n    if [ ! -f \"$PID_FILE\" ]; then\n        echo \"\u753b\u9762\u53ce\u9332\u306f\u5b9f\u884c\u3055\u308c\u3066\u3044\u307e\u305b\u3093\u3002\"\n        return 0\n    fi\n\n    REC_PID=$(cat \"$PID_FILE\")\n    if kill -0 \"$REC_PID\" 2>/dev/null; then\n        echo \">> \u753b\u9762\u53ce\u9332\u3092\u7d42\u4e86\u30fbMP4\u30d8\u30c3\u30c0\u30fc\u3092\u78ba\u5b9a\u4e2d (PID: $REC_PID)...\"\n        kill -INT \"$REC_PID\" 2>/dev/null || true\n        # \u7d42\u4e86\u5f85\u6a5f\n        for i in {1..10}; do\n            if ! kill -0 \"$REC_PID\" 2>/dev/null; then\n                break\n            fi\n            sleep 1\n        done\n        echo \"\u2705 \u753b\u9762\u53ce\u9332\u3092\u6b63\u5e38\u306b\u7d42\u4e86\u3057\u307e\u3057\u305f\u3002\"\n    fi\n    rm -f \"$PID_FILE\"\n\n    # Google Drive \u3078\u306e\u540c\u671f\n    CURRENT_FILE=$(cat /content/recorder_current_file.txt 2>/dev/null || true)\n    if [ -n \"$CURRENT_FILE\" ] && [ -f \"$CURRENT_FILE\" ]; then\n        echo \"\ud83d\udcc1 \u9332\u753b\u30d5\u30a1\u30a4\u30eb: $CURRENT_FILE ($(ls -lh \"$CURRENT_FILE\" | awk '{print $5}'))\"\n        if [ -d \"/content/drive/MyDrive\" ]; then\n            mkdir -p \"$GDRIVE_REC\"\n            cp -f \"$CURRENT_FILE\" \"$GDRIVE_REC/\" 2>/dev/null || true\n            echo \"\u2705 Google Drive ($GDRIVE_REC) \u306b\u540c\u671f\u3057\u307e\u3057\u305f\u3002\"\n        fi\n    fi\n}\n\nstatus_recording() {\n    if [ -f \"$PID_FILE\" ] && kill -0 \"$(cat \"$PID_FILE\")\" 2>/dev/null; then\n        CURRENT_FILE=$(cat /content/recorder_current_file.txt 2>/dev/null || true)\n        echo \"\ud83d\udd34 \u753b\u9762\u53ce\u9332\u4e2d (PID: $(cat \"$PID_FILE\"), \u30d5\u30a1\u30a4\u30eb: $CURRENT_FILE)\"\n    else\n        echo \"\u26aa \u753b\u9762\u53ce\u9332\u306f\u505c\u6b62\u3057\u3066\u3044\u307e\u3059\u3002\"\n    fi\n}\n\ncase \"$ACTION\" in\n    start)\n        start_recording\n        ;;\n    stop)\n        stop_recording\n        ;;\n    status)\n        status_recording\n        ;;\n    toggle)\n        if [ -f \"$PID_FILE\" ] && kill -0 \"$(cat \"$PID_FILE\")\" 2>/dev/null; then\n            stop_recording\n        else\n            start_recording\n        fi\n        ;;\n    *)\n        echo \"Usage: $0 {start|stop|status|toggle}\"\n        exit 1\n        ;;\nesac\n",
    "scripts/backup_restore.py": "#!/usr/bin/env python3\n\"\"\"\nbackup_restore.py - Google Drive Save Data & Profile Synchronization for Gaming PC\n(Syncs Minecraft worlds, Steam saves, Prism Launcher instances, Chrome profiles, and recordings)\n\"\"\"\n\nimport os\nimport sys\nimport shutil\nimport tarfile\nfrom datetime import datetime\nfrom pathlib import Path\n\nGDRIVE_BASE = Path(\"/content/drive/MyDrive/GamingPC\")\nGDRIVE_RECORDINGS = GDRIVE_BASE / \"recordings\"\n\n# \u30d0\u30c3\u30af\u30a2\u30c3\u30d7\u5bfe\u8c61\u30d1\u30b9\u306e\u5b9a\u7fa9 (\u5b58\u5728\u3059\u308b\u5834\u5408\u306e\u307f\u30a2\u30fc\u30ab\u30a4\u30d6)\nTARGETS = [\n    {\"src\": Path(\"/content/minecraft_server/world\"), \"arcname\": \"minecraft_server/world\"},\n    {\"src\": Path(\"/content/minecraft_server/world_nether\"), \"arcname\": \"minecraft_server/world_nether\"},\n    {\"src\": Path(\"/content/minecraft_server/world_the_end\"), \"arcname\": \"minecraft_server/world_the_end\"},\n    {\"src\": Path(\"/content/minecraft_server/server.properties\"), \"arcname\": \"minecraft_server/server.properties\"},\n    {\"src\": Path(\"/content/minecraft_server/plugins\"), \"arcname\": \"minecraft_server/plugins\"},\n    {\"src\": Path(\"/content/chrome_profiles\"), \"arcname\": \"chrome_profiles\"},\n    {\"src\": Path.home() / \".local/share/PrismLauncher/instances\", \"arcname\": \"prismlauncher_instances\"},\n    {\"src\": Path.home() / \".steam/steam/userdata\", \"arcname\": \"steam_userdata\"},\n    {\"src\": Path.home() / \".config/discord\", \"arcname\": \"config_discord\"},\n]\n\ndef log(msg: str):\n    print(f\"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}\", flush=True)\n\ndef restore_from_storage():\n    \"\"\"Google Drive\u304b\u3089\u30bb\u30fc\u30d6\u30c7\u30fc\u30bf\u30fb\u30d7\u30ed\u30d5\u30a1\u30a4\u30eb\u3092\u5fa9\u5143\"\"\"\n    if not GDRIVE_BASE.exists():\n        log(\"Google Drive\u306e\u30de\u30a6\u30f3\u30c8\u307e\u305f\u306f\u30d0\u30c3\u30af\u30a2\u30c3\u30d7\u30c7\u30a3\u30ec\u30af\u30c8\u30ea\u304c\u898b\u3064\u304b\u308a\u307e\u305b\u3093\u3002\u65b0\u898f\u74b0\u5883\u3068\u3057\u3066\u958b\u59cb\u3057\u307e\u3059\u3002\")\n        return False\n\n    latest_archive = GDRIVE_BASE / \"gaming_data_latest.tar.gz\"\n    if not latest_archive.exists():\n        # \u30a2\u30fc\u30ab\u30a4\u30d6\u4e00\u89a7\u304b\u3089\u6700\u65b0\u3092\u691c\u7d22\n        archives = sorted(GDRIVE_BASE.glob(\"gaming_data_*.tar.gz\"), key=os.path.getmtime, reverse=True)\n        if archives:\n            latest_archive = archives[0]\n\n    if latest_archive.exists():\n        log(f\"\ud83d\udce6 Google Drive\u304b\u3089\u6700\u65b0\u30d0\u30c3\u30af\u30a2\u30c3\u30d7\u3092\u5fa9\u5143\u4e2d: {latest_archive.name}\")\n        with tarfile.open(latest_archive, \"r:gz\") as tar:\n            for member in tar.getmembers():\n                dest_base = None\n                rel_path = None\n\n                if member.name.startswith(\"minecraft_server/\"):\n                    dest_base = Path(\"/content/minecraft_server\")\n                    rel_path = member.name[len(\"minecraft_server/\"):]\n                elif member.name.startswith(\"chrome_profiles\"):\n                    dest_base = Path(\"/content\")\n                    rel_path = member.name\n                elif member.name.startswith(\"prismlauncher_instances\"):\n                    dest_base = Path.home() / \".local/share/PrismLauncher\"\n                    rel_path = \"instances\" + member.name[len(\"prismlauncher_instances\"):]\n                elif member.name.startswith(\"steam_userdata\"):\n                    dest_base = Path.home() / \".steam/steam\"\n                    rel_path = \"userdata\" + member.name[len(\"steam_userdata\"):]\n                elif member.name.startswith(\"config_discord\"):\n                    dest_base = Path.home() / \".config\"\n                    rel_path = \"discord\" + member.name[len(\"config_discord\"):]\n\n                if dest_base and rel_path:\n                    target_file = dest_base / rel_path\n                    target_file.parent.mkdir(parents=True, exist_ok=True)\n                    if member.isdir():\n                        target_file.mkdir(parents=True, exist_ok=True)\n                    else:\n                        f = tar.extractfile(member)\n                        if f:\n                            with open(target_file, \"wb\") as out:\n                                shutil.copyfileobj(f, out)\n\n        log(\"\u2705 \u30bb\u30fc\u30d6\u30c7\u30fc\u30bf\u30fb\u30d7\u30ed\u30d5\u30a1\u30a4\u30eb\u306e\u5fa9\u5143\u304c\u5b8c\u4e86\u3057\u307e\u3057\u305f\u3002\")\n        return True\n\n    log(\"\u5fa9\u5143\u5bfe\u8c61\u306e\u30a2\u30fc\u30ab\u30a4\u30d6\u304c\u3042\u308a\u307e\u305b\u3093\u3067\u3057\u305f\u3002\")\n    return False\n\ndef backup_to_storage():\n    \"\"\"\u30bb\u30fc\u30d6\u30c7\u30fc\u30bf\u30fb\u30d7\u30ed\u30d5\u30a1\u30a4\u30eb\u30fb\u9332\u753b\u52d5\u753b\u3092Google Drive\u3078\u5b8c\u5168\u540c\u671f\"\"\"\n    if not Path(\"/content/drive/MyDrive\").exists():\n        log(\"\u26a0\ufe0f Google Drive\u304c\u30de\u30a6\u30f3\u30c8\u3055\u308c\u3066\u3044\u306a\u3044\u305f\u3081\u3001Drive\u540c\u671f\u3092\u30b9\u30ad\u30c3\u30d7\u3057\u307e\u3059\u3002\")\n        return None\n\n    GDRIVE_BASE.mkdir(parents=True, exist_ok=True)\n    GDRIVE_RECORDINGS.mkdir(parents=True, exist_ok=True)\n\n    timestamp = datetime.now().strftime(\"%Y%m%d_%H%M%S\")\n    archive_name = f\"gaming_data_{timestamp}.tar.gz\"\n    archive_path = GDRIVE_BASE / archive_name\n    latest_path = GDRIVE_BASE / \"gaming_data_latest.tar.gz\"\n\n    log(f\"\ud83d\udce6 \u30bb\u30fc\u30d6\u30c7\u30fc\u30bf\u3068\u8a2d\u5b9a\u3092\u5727\u7e2e\u30d0\u30c3\u30af\u30a2\u30c3\u30d7\u4e2d: {archive_name}\")\n    has_items = False\n    with tarfile.open(archive_path, \"w:gz\") as tar:\n        for t in TARGETS:\n            src = t[\"src\"]\n            arcname = t[\"arcname\"]\n            if src.exists():\n                tar.add(src, arcname=arcname)\n                has_items = True\n\n    if has_items:\n        shutil.copy2(archive_path, latest_path)\n        log(\"\u2705 \u30bb\u30fc\u30d6\u30c7\u30fc\u30bf\u30a2\u30fc\u30ab\u30a4\u30d6\u3092Google Drive\u3078\u4fdd\u5b58\u3057\u307e\u3057\u305f\u3002\")\n    else:\n        log(\"\u4fdd\u5b58\u5bfe\u8c61\u306e\u30c7\u30fc\u30bf\u304c\u307e\u3060\u3042\u308a\u307e\u305b\u3093\u3002\")\n\n    # \u753b\u9762\u53ce\u9332\u52d5\u753b (MP4) \u306e\u540c\u671f\n    local_rec = Path(\"/content/recordings\")\n    if local_rec.exists():\n        for mp4 in local_rec.glob(\"*.mp4\"):\n            dest_mp4 = GDRIVE_RECORDINGS / mp4.name\n            if not dest_mp4.exists() or dest_mp4.stat().st_size != mp4.stat().st_size:\n                log(f\"\ud83c\udfa5 \u9332\u753b\u30d5\u30a1\u30a4\u30eb\u3092\u540c\u671f\u4e2d: {mp4.name} ({mp4.stat().st_size / (1024*1024):.1f} MB)\")\n                shutil.copy2(mp4, dest_mp4)\n        log(\"\u2705 \u753b\u9762\u53ce\u9332\u52d5\u753b\u306eGoogle Drive\u540c\u671f\u5b8c\u4e86\u3002\")\n\n    return latest_path\n\ndef backup_status():\n    \"\"\"\u73fe\u5728\u306e\u30d0\u30c3\u30af\u30a2\u30c3\u30d7\u72b6\u6cc1\u3092\u78ba\u8a8d\"\"\"\n    if not GDRIVE_BASE.exists():\n        log(\"Google Drive\u540c\u671f\u30c7\u30a3\u30ec\u30af\u30c8\u30ea\u304c\u5b58\u5728\u3057\u307e\u305b\u3093\u3002\")\n        return\n\n    archives = list(GDRIVE_BASE.glob(\"gaming_data_*.tar.gz\"))\n    recordings = list(GDRIVE_RECORDINGS.glob(\"*.mp4\")) if GDRIVE_RECORDINGS.exists() else []\n\n    log(f\"Google Drive\u4e0a\u306e\u30d0\u30c3\u30af\u30a2\u30c3\u30d7\u6570: {len(archives)} \u500b\")\n    for a in sorted(archives, key=os.path.getmtime, reverse=True)[:3]:\n        log(f\"  - {a.name} ({a.stat().st_size / (1024*1024):.2f} MB)\")\n\n    log(f\"Google Drive\u4e0a\u306e\u9332\u753b\u52d5\u753b\u6570: {len(recordings)} \u672c\")\n    for r in sorted(recordings, key=os.path.getmtime, reverse=True)[:3]:\n        log(f\"  - {r.name} ({r.stat().st_size / (1024*1024):.2f} MB)\")\n\nif __name__ == \"__main__\":\n    action = sys.argv[1] if len(sys.argv) > 1 else \"backup\"\n    if action == \"restore\":\n        restore_from_storage()\n    elif action == \"backup\":\n        backup_to_storage()\n    elif action == \"status\":\n        backup_status()\n    else:\n        print(f\"Unknown action: {action}\")\n",
    "scripts/shutdown.sh": "#!/bin/bash\n# ==============================================================================\n# shutdown.sh - Safe Shutdown & Complete Data Synchronization\n# (Stops recording, saves Minecraft, syncs to Google Drive, terminates daemons)\n# ==============================================================================\n\nREPO_DIR=\"$(cd \"$(dirname \"${BASH_SOURCE[0]}\")/..\" && pwd)\"\nSERVER_DIR=\"/content/minecraft_server\"\nPIPE_PATH=\"$SERVER_DIR/server.pipe\"\n\necho \"==================================================\"\necho \"  \ud83d\uded1 Gaming PC - \u5b89\u5168\u306a\u30b7\u30e3\u30c3\u30c8\u30c0\u30a6\u30f3\u51e6\u7406\u958b\u59cb\"\necho \"==================================================\"\n\n# 1. \u753b\u9762\u53ce\u9332\u30d7\u30ed\u30bb\u30b9\u306e\u505c\u6b62\necho \">> [1/5] \u753b\u9762\u53ce\u9332\u30d7\u30ed\u30bb\u30b9\u306e\u7d42\u4e86\u3068MP4\u306e\u78ba\u5b9a...\"\nbash \"$REPO_DIR/scripts/record_screen.sh\" stop 2>/dev/null || true\n\n# 2. \u5b9a\u671f\u30d0\u30c3\u30af\u30a2\u30c3\u30d7\u30d7\u30ed\u30bb\u30b9\u306e\u7d42\u4e86\npkill -f \"backup_restore.py\" 2>/dev/null || true\n\n# 3. Minecraft \u30b5\u30fc\u30d0\u30fc\u306e\u30ef\u30fc\u30eb\u30c9\u4fdd\u5b58\u3068\u5b89\u5168\u306a\u505c\u6b62\necho \">> [2/5] Minecraft\u30b5\u30fc\u30d0\u30fc\u306e\u30ef\u30fc\u30eb\u30c9\u4fdd\u5b58 (save-all) & \u505c\u6b62...\"\nif [ -p \"$PIPE_PATH\" ] && pgrep -f \"server.jar\" > /dev/null 2>&1; then\n    echo \"say \u30b5\u30fc\u30d0\u30fc\u3092\u5b89\u5168\u306b\u30b7\u30e3\u30c3\u30c8\u30c0\u30a6\u30f3\u3057\u307e\u3059...\" > \"$PIPE_PATH\" 2>/dev/null || true\n    sleep 1\n    echo \"save-all\" > \"$PIPE_PATH\" 2>/dev/null || true\n    sleep 2\n    echo \"stop\" > \"$PIPE_PATH\" 2>/dev/null || true\n    sleep 3\nfi\n\n# \u30b5\u30fc\u30d0\u30fc\u30d7\u30ed\u30bb\u30b9\u7d42\u4e86\u5f85\u6a5f\nfor i in {1..15}; do\n    if ! pgrep -f \"server.jar\" > /dev/null 2>&1; then\n        break\n    fi\n    sleep 1\n    echo -n \".\"\ndone\necho \"\"\npkill -TERM -f \"server.jar\" 2>/dev/null || true\nrm -f \"$PIPE_PATH\" /content/mc_server.pid\n\n# 4. \u30bb\u30fc\u30d6\u30c7\u30fc\u30bf & \u9332\u753b\u52d5\u753b\u306eGoogle Drive\u540c\u671f\necho \">> [3/5] \u30bb\u30fc\u30d6\u30c7\u30fc\u30bf\u30fb\u30d7\u30ed\u30d5\u30a1\u30a4\u30eb\u30fb\u9332\u753b\u52d5\u753b\u306e\u6700\u7d42Google Drive\u540c\u671f...\"\npython3 \"$REPO_DIR/scripts/backup_restore.py\" backup || true\n\n# 5. \u30c8\u30f3\u30cd\u30eb\u304a\u3088\u3073\u30c7\u30b9\u30af\u30c8\u30c3\u30d7\u30d7\u30ed\u30bb\u30b9\u306e\u7d42\u4e86\necho \">> [4/5] \u30c8\u30f3\u30cd\u30eb & \u30ea\u30e2\u30fc\u30c8\u30c7\u30b9\u30af\u30c8\u30c3\u30d7\u30d7\u30ed\u30bb\u30b9\u306e\u505c\u6b62...\"\npkill -f \"playit\" 2>/dev/null || true\npkill -f \"cloudflared tunnel\" 2>/dev/null || true\npkill -f \"websockify\" 2>/dev/null || true\npkill -f \"x11vnc\" 2>/dev/null || true\npkill -f \"xfce4\" 2>/dev/null || true\npkill -f \"Xvfb :1\" 2>/dev/null || true\n\necho \">> [5/5] \u30b7\u30e3\u30c3\u30c8\u30c0\u30a6\u30f3\u5b8c\u4e86\u78ba\u8a8d...\"\necho \"\"\necho \"==================================================\"\necho \"\ud83c\udf89 \u3059\u3079\u3066\u306e\u30c7\u30fc\u30bf\u4fdd\u5b58\u3068\u30b7\u30e3\u30c3\u30c8\u30c0\u30a6\u30f3\u304c\u5b8c\u4e86\u3057\u307e\u3057\u305f\uff01\"\necho \"\ud83d\udcbe \u4fdd\u5b58\u5148: Google Drive > GamingPC/\"\necho \"\u304a\u75b2\u308c\u69d8\u3067\u3057\u305f\u3002\"\necho \"==================================================\"\n"
}

for rel_path, content in EMBEDDED_FILES.items():
    target = os.path.join(REPO_DIR, rel_path)
    os.makedirs(os.path.dirname(target), exist_ok=True)
    with open(target, "w", encoding="utf-8") as f:
        f.write(content)
    os.chmod(target, 0o755)

# 3. Google Drive からのセーブデータ復元
print(">> [1/4] Google Drive からのセーブデータ・プロファイル復元チェック...")
subprocess.run([sys.executable, os.path.join(SCRIPTS_DIR, "backup_restore.py"), "restore"], check=False)

# 4. 環境セットアップの実行
print(">> [2/4] 仮想環境セットアップ (日本語化・Steam・Minecraft・Chrome)... 約60秒")
subprocess.run(["bash", os.path.join(SCRIPTS_DIR, "setup_environment.sh")], check=True)

# 5. デスクトップ & リアルタイム音同期の起動
print(">> [3/4] Xfce4 デスクトップ & Web Audio 超低遅延音同期エンジンの起動...")
crd_arg = CRD_AUTH_CODE if "Chrome Remote Desktop" in REMOTE_METHOD else ""
subprocess.run(["bash", os.path.join(SCRIPTS_DIR, "start_desktop.sh"), crd_arg, CRD_PIN, SCREEN_RES], check=True)

# 6. Minecraft サーバーの独立バックグラウンド起動
if START_MINECRAFT_SERVER:
    print(">> [4/4] 独立Minecraftサーバー (PaperMC 1.21.1) をバックグラウンド起動中...")
    subprocess.run(["bash", os.path.join(SCRIPTS_DIR, "start_mc_server.sh"), str(MC_RAM_GB)], check=True)
else:
    print(">> [4/4] Minecraftサーバーの起動はスキップされました。")

# 7. 接続リンクの生成と表示
colab_base = eval_js("google.colab.kernel.proxyPort(6080)").rstrip("/")
vnc_url = f"{colab_base}/vnc.html?autoconnect=true&resize=scale"

cf_url = ""
try:
    with open("/content/cloudflared.log", "r") as f:
        for line in f:
            if ".trycloudflare.com" in line:
                import re
                m = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
                if m:
                    cf_url = f"{m.group(0)}/vnc.html?autoconnect=true&resize=scale"
                    break
except Exception:
    pass

display(HTML(f'''
<div style="background: #0f172a; padding: 24px; border-radius: 12px; border: 1px solid #334155; border-left: 6px solid #10b981; font-family: sans-serif; box-shadow: 0 4px 15px rgba(0,0,0,0.4); margin-top: 15px;">
  <h3 style="color: #10b981; margin: 0 0 10px 0; font-size: 20px; display: flex; align-items: center; gap: 8px;">
    🎉 Gaming PC 起動完了！（日本語・音同期・MacBook 16:10 最適化）
  </h3>
  <p style="color: #cbd5e1; font-size: 14px; margin-bottom: 18px;">
    下のボタンをクリックすると、ブラウザの新しいタブで <strong>日本語デスクトップ</strong> が開きます：
  </p>
  <div style="display: flex; gap: 12px; flex-wrap: wrap; margin-bottom: 16px;">
    <a href="{vnc_url}" target="_blank" style="display: inline-block; background: #10b981; color: #042f2e; padding: 14px 28px; border-radius: 8px; font-size: 16px; font-weight: bold; text-decoration: none; box-shadow: 0 4px 12px rgba(16, 185, 129, 0.3);">
      🚀 今すぐデスクトップを開く (Colabプロキシ)
    </a>
    {f'<a href="{cf_url}" target="_blank" style="display: inline-block; background: #f97316; color: #fff; padding: 14px 28px; border-radius: 8px; font-size: 16px; font-weight: bold; text-decoration: none;">⚡ Cloudflare 高速トンネルで開く</a>' if cf_url else ''}
  </div>
  <p style="font-size: 12px; color: #94a3b8; margin: 0;">
    ※ デスクトップ画面内を一度クリックすると、右上に <strong>[LIVE SOUND]</strong> が表示され、YouTube やゲームの音が Mac からリアルタイムに再生されます。
  </p>
</div>
'''))


---
## ⛏️ ステップ 2: Minecraft サーバーの管理 (任意)

Minecraft サーバーはステップ 1 で既にバックグラウンド起動していますが、OP権限付与やコマンド投入、ログ確認はこちらのセルで行えます。

In [ ]:
#@title ⛏️ ステップ 2: Minecraft サーバー コマンド投入 & ログ確認
import os

#@markdown ### 🕹️ 実行したいコマンド (例: `op your_name`, `whitelist on`, `gamemode creative`)
SERVER_CMD = "list" #@param {type:"string"}

PIPE_PATH = "/content/minecraft_server/server.pipe"
if os.path.exists(PIPE_PATH) and SERVER_CMD.strip():
    with open(PIPE_PATH, "w") as pipe:
        pipe.write(SERVER_CMD.strip() + "\n")
    print(f"✅ コマンド送信完了: {SERVER_CMD}")
else:
    print("⚠️ サーバーが停止しているか、パイプが見つかりません。")

print("\n--- サーバー最新ログ (最新15行) ---")
if os.path.exists("/content/minecraft_server/logs/latest.log"):
    with open("/content/minecraft_server/logs/latest.log", "r") as f:
        lines = f.readlines()
        print("".join(lines[-15:]))


---
## 🎥 ステップ 3: 画面収録 (L4 GPU NVENC 60fps) コントロール

NVIDIA L4 GPU のハードウェアエンコーダーを活用し、ゲームプレイや作業を 1080p 60fps で録画します。  
（デスクトップ上の「画面収録」アイコンからでも操作可能です）

In [ ]:
#@title 🎥 ステップ 3: 画面収録コントロール
import subprocess

#@markdown ### 🎬 操作の選択
RECORDER_ACTION = "status" #@param ["status", "start", "stop"]

subprocess.run(["bash", "/content/ColabTool/GamingPC/scripts/record_screen.sh", RECORDER_ACTION])


---
## 💾 ステップ 4: Google Drive 即時同期 (手動バックアップ)

Minecraft ワールド、Steam セーブ、Chrome プロファイル、録画動画を今すぐ Google Drive へ同期します。

In [ ]:
#@title 💾 ステップ 4: Google Drive 即時同期
import subprocess
import sys

subprocess.run([sys.executable, "/content/ColabTool/GamingPC/scripts/backup_restore.py", "backup"])
subprocess.run([sys.executable, "/content/ColabTool/GamingPC/scripts/backup_restore.py", "status"])


---
## 🛑 ステップ 5: 安全なシャットダウン＆データ保存

作業終了時にこのセルを実行してください。  
Minecraft サーバーのワールド保存（`save-all`）、画面収録の確定、Google Drive への最終同期、全プロセスの安全な停止が全自動で行われます。

In [ ]:
#@title 🛑 ステップ 5: 安全なシャットダウン＆データ保存
import subprocess

subprocess.run(["bash", "/content/ColabTool/GamingPC/scripts/shutdown.sh"])
